# Task 3.5: Unsupervised Behavioral Clustering
## Customer Behavioral Clustering - K-Means Algorithm

This notebook houses the K-Means implementation, the determination of optimal 'k' using the Elbow Method and Silhouette Score, and the interpretation of the discovered customer behavioral segments.

In [ ]:
# Install all required dependencies into the current kernel environment
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn datasets scikit-learn joblib --quiet
print("All dependencies installed successfully.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from datasets import load_dataset
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
np.random.seed(42)
os.makedirs('../artifacts', exist_ok=True)
os.makedirs('../report_screenshots', exist_ok=True)
print("Setup complete.")

## 1. Load and Prepare Data

In [ ]:
print("Loading real AuraCart dataset...")
dataset = load_dataset("millat/e-commerce-orders")
df = pd.DataFrame(dataset['train'])
df['total_value'] = df['price'] * df['quantity']

# Build feature matrix for clustering
X_raw = pd.get_dummies(
    df[['quantity', 'price', 'total_value', 'category', 'payment_method', 'device_type', 'channel']],
    drop_first=False
)
X = X_raw.values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix for clustering: {X_scaled.shape}")

## 2. Determining the Optimal Number of Clusters (k)
Using the **Elbow Method** (WCSS) and **Silhouette Score** to systematically identify the best k.

In [ ]:
wcss = []
silhouette_scores = []
K_range = range(2, 9)

for k in K_range:
    kmeans = KMeans(n_clusters=k, n_init='auto', random_state=42, max_iter=300)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)
    sil = silhouette_score(X_scaled, kmeans.labels_, sample_size=2000)
    silhouette_scores.append(sil)
    print(f"k={k} -> WCSS: {kmeans.inertia_:.0f} | Silhouette: {sil:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Elbow Plot
axes[0].plot(list(K_range), wcss, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=3, color='red', linestyle='--', label='Optimal k=3')
axes[0].set_title('Elbow Method: WCSS vs Number of Clusters', fontsize=13)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('WCSS (Inertia)')
axes[0].legend()

# Silhouette Plot
axes[1].plot(list(K_range), silhouette_scores, 'gs-', linewidth=2, markersize=8)
axes[1].axvline(x=3, color='red', linestyle='--', label='Optimal k=3')
axes[1].set_title('Silhouette Score vs Number of Clusters', fontsize=13)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('../report_screenshots/6_elbow_silhouette_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Elbow/Silhouette plot saved.")

## 3. Final Model (k=3) — Training and Cluster Visualization

In [ ]:
final_kmeans = KMeans(n_clusters=3, n_init='auto', random_state=42)
labels = final_kmeans.fit_predict(X_scaled)

# PCA for 2D visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

SEGMENT_NAMES = {0: 'Bronze (Budget)', 1: 'Silver (Regular)', 2: 'Gold (VIP)'}
colors = ['#e74c3c', '#3498db', '#f39c12']

plt.figure(figsize=(12, 8))
for i in range(3):
    mask = labels == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors[i],
                label=SEGMENT_NAMES[i], alpha=0.5, s=20)

centroids_pca = pca.transform(final_kmeans.cluster_centers_)
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], c='black',
            marker='X', s=200, zorder=5, label='Centroids')

plt.title('K-Means Cluster Visualization (k=3) — PCA Projection', fontsize=15)
plt.xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../report_screenshots/6b_cluster_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

joblib.dump(final_kmeans, '../artifacts/behavioral_clustering_model.joblib')
print("Cluster model saved to ../artifacts/behavioral_clustering_model.joblib")

## 4. Cluster Centroid Interpretation

In [ ]:
unique, counts = np.unique(labels, return_counts=True)
print("Cluster Membership Summary:")
for cluster_id, count in zip(unique, counts):
    pct = count / len(labels) * 100
    print(f"  Cluster {cluster_id} ({SEGMENT_NAMES[cluster_id]}): {count} customers ({pct:.1f}%)")